In [1]:
import os
import urllib.request

import numpy as np

import astropy.units as u

from astropy.io import fits
from astropy.coordinates import SkyCoord
from astroquery.heasarc import Heasarc

heasarc = Heasarc()

coma_coord = SkyCoord(ra=194.953, dec=27.961, unit=(u.deg, u.deg), frame='icrs')
pers_coord = SkyCoord(ra=49.948, dec=41.510, unit=(u.deg, u.deg), frame='icrs')

In [2]:
def retrieve(coord):
    print('searching chandra catalog in given region...\n')

    results = heasarc.query_region(coord, catalog='chanmaster', radius=15 * u.arcmin)
    results.sort('exposure')
    results.reverse()

    print('results from chandra catalog in given region:\n')

    print(results['obsid', 'exposure', 'ra', 'dec'][:10])
    
    best_obsid = str(results['obsid'][0])

    base_url = f'https://heasarc.gsfc.nasa.gov/FTP/chandra/data/byobsid/{best_obsid[-1]}/{best_obsid}/primary/'

    new_filename = f'acisf{best_obsid.zfill(5)}N004_evt2.fits.gz'
    old_filename = f'acisf{best_obsid.zfill(5)}N003_evt2.fits.gz'

    download = True

    if os.path.exists('chandra/' + new_filename):
        filename = new_filename
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('chandra/' + old_filename):
        filename = old_filename
        print('\nmatching file found, skipping download.\n')
    else:
        try:
            print('attempting download...\n')
            filename = new_filename
            download_url = base_url + filename
            urllib.request.urlretrieve(full_download_url, 'chandra/' + filename)
        except:
            try:
                print('file not found, trying fallback...\n')
                filename = old_filename
                download_url = base_url + filename
                urllib.request.urlretrieve(download_url, 'chandra/' + filename)
            except:
                download = False
                print('download failed, aborting...')

    if download:
        print('extracting binned spectrum...\n')
        with fits.open('chandra/' + filename) as hdul:
            data = hdul[1].data
            header = hdul[1].header
        
            exposure = header.get('EXPOSURE', 1.0)
        
            energies = data['energy'] / 1000.0
        
            edges = np.linspace(2.0, 10.0, 1001)
            counts, _ = np.histogram(energies, bins=edges)
        
            centers = (edges[:-1] + edges[1:]) / 2.0
            widths = np.diff(edges)
            
            flux = counts / (exposure * widths)
            error = np.sqrt(counts) / (exposure * widths)
        
            mask = flux > 0
        
            centers = centers[mask]
            flux = flux[mask]
            error = error[mask]
    
            output_file = f'{filename[:-8]}.txt'
            
            np.savetxt('chandra/' + output_file, np.column_stack((centers, flux, error)), header='E / keV # rel_F # rel_F_err #')
        print('done.')

In [3]:
retrieve(coma_coord)

searching chandra catalog in given region...

results from chandra catalog in given region:

obsid exposure     ra      dec   
         s        deg      deg   
----- -------- --------- --------
13996   124680 194.96256 27.94269
13994    83080 194.91427 27.90794
14410    79570 194.96635 27.90493
13995    63820 194.99819 27.91735
13993    40080 194.96635 27.90493
14415    34970 194.99819 27.91735
14411    34080 194.91427 27.90794
18235    30070 195.15917 28.04906
 9714    30040 194.95000 27.96667
10672    28910 194.95000 27.96667

matching file found, skipping download.

extracting binned spectrum...

done.


In [4]:
retrieve(pers_coord)

searching chandra catalog in given region...

results from chandra catalog in given region:

obsid exposure    ra      dec   
         s       deg      deg   
----- -------- -------- --------
 4952   166420 49.95083 41.51172
 4948   120180 49.95083 41.51172
11713   113680 49.88250 41.63028
 4950    98200 49.95083 41.51172
 4951    97390 49.95083 41.51172
 3209    97040 49.94833 41.51028
 4289    96680 49.94833 41.51028
11714    93210 49.92750 41.56861
 6145    86130 49.95083 41.51172
12037    85760 49.93417 41.42167

matching file found, skipping download.

extracting binned spectrum...

done.
